# Torrent-to-Drive on Google Colab

Runs the full app (backend + web UI) on a free Colab VM and uploads torrents straight to your Google Drive.

**How to use:** run the cells top to bottom (`Runtime → Run all` works, the login cell will pause for you).

| Step | What happens |
|---|---|
| 1. Settings | Pick the repo branch and the Drive folder to upload into |
| 2. Install | Clones the repo, installs Python deps + Node 20, builds the UI (~2–3 min) |
| 3. Log in | Google sign-in popup; the app reuses this login for Drive (no OAuth client / `credentials.json` needed) |
| 4. Start | Starts the server in the background |
| 5. Open UI | Opens the UI through Colab's private proxy (only you can reach it) |

**Good to know**
- Colab disk is ~100 GB and the VM is wiped when the runtime ends. For torrents bigger than free disk, turn on **Settings → Stream to Drive** in the UI: it downloads in order, uploads each chunk, then frees it.
- Free Colab disconnects after ~90 min idle / ~12 h max. Interrupted torrents show as **Paused**; streamed uploads continue from the last chunk if you re-run on the same VM. On a fresh VM the download restarts (local data is gone) but files already on Drive are skipped.
- Only download content you have the right to download. Colab's terms forbid some uses (e.g. P2P file sharing on the free tier may get the runtime stopped).

## 1. Settings

In [ ]:
REPO_URL = "https://github.com/hencydsouza/torrent-to-drive.git"  #@param {type:"string"}
BRANCH = "feat/streaming-settings-qol"  #@param {type:"string"}
# Folder ID from the Drive URL: drive.google.com/drive/folders/<THIS_PART>. "root" = top of My Drive.
DRIVE_FOLDER_ID = "root"  #@param {type:"string"}
STREAM_TO_DRIVE = False  #@param {type:"boolean"}

## 2. Install (2–3 min)

In [ ]:
%cd /content
!rm -rf torrent-to-drive && git clone -q --depth 1 -b "$BRANCH" "$REPO_URL" torrent-to-drive
%cd /content/torrent-to-drive
!pip install -q -r requirements.txt
# Node 20 to build the web UI (Colab's preinstalled Node may be older)
!node -v 2>/dev/null | grep -q '^v2[0-9]' || (curl -fsSL https://deb.nodesource.com/setup_20.x | bash - >/dev/null && apt-get install -y -qq nodejs >/dev/null)
!cd frontend && npm ci --silent && npm run build --silent
!python -c "import libtorrent as lt; print('libtorrent', lt.__version__)"

## 3. Log in to Google
A popup asks you to sign in and allow access. The app uses this login to upload to your Drive.

In [ ]:
from google.colab import auth
auth.authenticate_user()

## 4. Start the server

In [ ]:
import os, subprocess, time, urllib.request
os.chdir("/content/torrent-to-drive")
env = dict(os.environ, GOOGLE_AUTH="adc", STREAM_TO_DRIVE=str(STREAM_TO_DRIVE).lower(), DRIVE_FOLDER_ID=DRIVE_FOLDER_ID or "root")
subprocess.run("pkill -f 'uvicorn backend.main:app'", shell=True)
server = subprocess.Popen("python -m uvicorn backend.main:app --host 0.0.0.0 --port 8000 > server.log 2>&1",
                          shell=True, env=env)
for _ in range(60):
    try:
        urllib.request.urlopen("http://localhost:8000/api/settings"); print("Server is up"); break
    except Exception:
        time.sleep(1)
else:
    print("Server did not start, last log lines:"); print(open("server.log").read()[-3000:])

## 5. Open the UI
Click the link below. It goes through Colab's proxy, so only your Google account can open it. Keep this tab open: closing the notebook eventually stops the VM.

In [ ]:
from google.colab.output import serve_kernel_port_as_window
serve_kernel_port_as_window(8000, path="/")

### Optional: public URL (if the Colab link doesn't load or live progress doesn't update)
Creates a temporary `*.trycloudflare.com` link. **Anyone with the link can control the app**, so don't share it.

In [ ]:
!test -f cloudflared || wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x cloudflared
import subprocess, re, time
tunnel = subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://localhost:8000"], stderr=subprocess.PIPE, text=True)
for line in tunnel.stderr:
    if m := re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line):
        print("Open:", m.group(0)); break

## Troubleshooting

In [ ]:
# Server log (last 50 lines)
!tail -n 50 /content/torrent-to-drive/server.log
# Free disk on the VM
!df -h /content | tail -1